# LLM-as-a-judge with repeatability and human calibration

Copyright (c) jphall@gwu.edu 2026-27. Released under the MIT License; see the repository `LICENSE` file.

This notebook treats an LLM judge as a measurement instrument. It exposes the rubric, requests structured ratings, repeats each judgment, and compares the results with small human reference ratings.

## 1. Configure the OpenAI client

The project-root `.env` is loaded first, with a secure `getpass` fallback.

In [1]:
# 1. Configure the OpenAI client
import json
import os
from collections import defaultdict
from getpass import getpass
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Locate the repository root from either the root or lecture directory.
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env').exists()), Path.cwd())
load_dotenv(project_root / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
client = OpenAI(api_key=api_key)
JUDGE_MODEL = os.getenv('OPENAI_JUDGE_MODEL', 'gpt-5-nano')
print(f'Configured judge model: {JUDGE_MODEL}')

Configured judge model: gpt-5-nano


## 2. Define the rubric and candidate answers

Every dimension has behavioral anchors. The candidate answers are fixed so repeat runs measure judge variability rather than answer-generation variability.

In [2]:
# 2. Define the rubric and candidate answers
question = 'In exactly two sentences, explain why held-out test data matters when evaluating an LLM application.'
candidates = {
    'strong': 'Held-out test data estimates how the application behaves on cases that were not used to tune the model, prompts, retrieval, or decision thresholds. Keeping it separate reduces optimistic bias, although representative sampling, contamination checks, uncertainty estimates, and field evidence are still needed.',
    'weak': 'Held-out data is important because testing is good. It proves the LLM will work for everyone.',
}
rubric = {
    'accuracy': {1: 'Contains a central error or unjustified claim.', 2: 'Mostly correct with a minor imprecision.', 3: 'Correct and appropriately qualified.'},
    'relevance': {1: 'Does not answer the question.', 2: 'Partly answers or adds distracting material.', 3: 'Directly answers the question.'},
    'completeness': {1: 'Omits the central reason.', 2: 'Gives the main reason but misses a useful limitation.', 3: 'Explains the main reason and material limitations.'},
    'clarity': {1: 'Confusing or contradictory.', 2: 'Understandable but imprecise.', 3: 'Clear and well organized.'},
    'instruction_following': {1: 'Violates the two-sentence requirement.', 2: 'Minor format deviation.', 3: 'Uses exactly two sentences.'},
}
# These reference ratings represent a small human calibration set, not universal truth.
human_ratings = {
    'strong': {'accuracy': 3, 'relevance': 3, 'completeness': 3, 'clarity': 3, 'instruction_following': 3},
    'weak': {'accuracy': 1, 'relevance': 2, 'completeness': 1, 'clarity': 2, 'instruction_following': 3},
}
print(question)
for name, answer in candidates.items():
    print(f'\n{name.upper()}: {answer}')

In exactly two sentences, explain why held-out test data matters when evaluating an LLM application.

STRONG: Held-out test data estimates how the application behaves on cases that were not used to tune the model, prompts, retrieval, or decision thresholds. Keeping it separate reduces optimistic bias, although representative sampling, contamination checks, uncertainty estimates, and field evidence are still needed.

WEAK: Held-out data is important because testing is good. It proves the LLM will work for everyone.


## 3. Create a strict structured-output schema

The schema prevents missing dimensions and out-of-range scores. It does not guarantee that the judgments are valid.

In [3]:
# 3. Create a strict structured-output schema
dimensions = list(rubric)
rating_schema = {
    'type': 'object',
    'properties': {
        'score': {'type': 'integer', 'minimum': 1, 'maximum': 3},
        'rationale': {'type': 'string'},
        'evidence': {'type': 'string'},
    },
    'required': ['score', 'rationale', 'evidence'],
    'additionalProperties': False,
}
judge_schema = {
    'type': 'object',
    'properties': {
        'ratings': {'type': 'object', 'properties': {name: rating_schema for name in dimensions}, 'required': dimensions, 'additionalProperties': False},
        'overall_feedback': {'type': 'string'},
        'uncertainty': {'type': 'string'},
    },
    'required': ['ratings', 'overall_feedback', 'uncertainty'],
    'additionalProperties': False,
}
print(json.dumps(judge_schema, indent=2)[:1200] + '\n...')

{
  "type": "object",
  "properties": {
    "ratings": {
      "type": "object",
      "properties": {
        "accuracy": {
          "type": "object",
          "properties": {
            "score": {
              "type": "integer",
              "minimum": 1,
              "maximum": 3
            },
            "rationale": {
              "type": "string"
            },
            "evidence": {
              "type": "string"
            }
          },
          "required": [
            "score",
            "rationale",
            "evidence"
          ],
          "additionalProperties": false
        },
        "relevance": {
          "type": "object",
          "properties": {
            "score": {
              "type": "integer",
              "minimum": 1,
              "maximum": 3
            },
            "rationale": {
              "type": "string"
            },
            "evidence": {
              "type": "string"
            }
          },
          "required":

## 4. Run each judgment twice

Repeated calls estimate small-change stability. The prompt separates instructions, rubric, question, and candidate answer with explicit markers.

In [4]:
# 4. Run each judgment twice
def judge_answer(candidate):
    prompt = f'''===== RUBRIC =====
{json.dumps(rubric, indent=2)}

===== QUESTION =====
{question}

===== CANDIDATE ANSWER =====
{candidate}
'''
    response = client.responses.create(
        model=JUDGE_MODEL,
        instructions='Apply every rubric dimension independently. Use only the displayed answer as evidence. Do not reward verbosity. State uncertainty rather than inventing support.',
        input=prompt,
        max_output_tokens=4000,
        reasoning={'effort': 'low'},
        text={'format': {'type': 'json_schema', 'name': 'tevV_judgment', 'strict': True, 'schema': judge_schema}},
        store=False,
    )
    if not response.output_text:
        # Surface an incomplete API response instead of failing later with an opaque JSON error.
        raise RuntimeError(f'Judge returned no visible JSON; status={response.status}, details={response.incomplete_details}')
    return json.loads(response.output_text)

judgments = defaultdict(list)
for candidate_name, answer in candidates.items():
    for repeat in range(2):
        judgment = judge_answer(answer)
        judgments[candidate_name].append(judgment)
        scores = {name: judgment['ratings'][name]['score'] for name in dimensions}
        print(candidate_name, 'repeat', repeat + 1, scores, 'total=', sum(scores.values()))

strong repeat 1 {'accuracy': 3, 'relevance': 3, 'completeness': 3, 'clarity': 3, 'instruction_following': 3} total= 15


strong repeat 2 {'accuracy': 3, 'relevance': 3, 'completeness': 3, 'clarity': 3, 'instruction_following': 3} total= 15


weak repeat 1 {'accuracy': 2, 'relevance': 3, 'completeness': 2, 'clarity': 2, 'instruction_following': 3} total= 12


weak repeat 2 {'accuracy': 2, 'relevance': 3, 'completeness': 2, 'clarity': 2, 'instruction_following': 3} total= 12


## 5. Measure repeatability and human agreement

Exact agreement is intentionally strict for this three-point rubric. With a larger calibration set, use dimension-level confusion matrices and an agreement statistic suited to ordinal ratings.

In [5]:
# 5. Measure repeatability and human agreement
repeat_matches = []
human_matches = []
for candidate_name in candidates:
    first, second = judgments[candidate_name]
    for dimension in dimensions:
        score_1 = first['ratings'][dimension]['score']
        score_2 = second['ratings'][dimension]['score']
        repeat_matches.append(score_1 == score_2)
        human_matches.extend([score_1 == human_ratings[candidate_name][dimension], score_2 == human_ratings[candidate_name][dimension]])
print(f'Exact repeat agreement: {np.mean(repeat_matches):.1%}')
print(f'Exact agreement with reference human ratings: {np.mean(human_matches):.1%}')
for candidate_name in candidates:
    totals = [sum(run['ratings'][d]['score'] for d in dimensions) for run in judgments[candidate_name]]
    print(f'{candidate_name:6s} totals: {totals}; mean={np.mean(totals):.1f}/15')

Exact repeat agreement: 100.0%
Exact agreement with reference human ratings: 70.0%
strong totals: [15, 15]; mean=15.0/15
weak   totals: [12, 12]; mean=12.0/15


## 6. Inspect rationales and limitations

Inspect the evidence behind every score; totals can conceal disagreement. A serious validation should use many held-out, independently human-rated cases; test order, verbosity, self-preference, prompt, model, and language effects; allow abstention; and define what agreement is sufficient for the intended decision.

In [6]:
# 6. Inspect rationales and limitations
for candidate_name, runs in judgments.items():
    print(f'\n=== {candidate_name.upper()} / FIRST RUN ===')
    for dimension in dimensions:
        rating = runs[0]['ratings'][dimension]
        print(f'{dimension:22s} {rating["score"]}: {rating["rationale"]} Evidence: {rating["evidence"]}')
    print('Uncertainty:', runs[0]['uncertainty'])


=== STRONG / FIRST RUN ===
accuracy               3: Statement accurately reflects why held-out data matters and is appropriately qualified by noting additional checks often used. Evidence: "Held-out test data...not used to tune the model...keeps optimistic bias down" with caveats about sampling and checks.
relevance              3: Directly answers why held-out data matters in evaluating an LLM application. Evidence: The sentence explains the role of held-out data in evaluation.
completeness           3: Gives the main reason and mentions relevant limitations and additional practices (sampling, contamination checks, uncertainty estimates, field evidence). Evidence: Two sentences cover why held-out data matters and caveats/limitations.
clarity                3: Clear and well organized; two concise sentences. Evidence: Two straightforward sentences with logical flow.
instruction_following  3: Uses exactly two sentences as required. Evidence: Two sentences present.
Uncertainty: low

==